In [26]:
%%capture
!pip install unsloth
# Also get the latest nightly Unsloth!
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
# unzip the model 
!unzip /content/llama_3B_LR2e5_DR0.zip

Archive:  /content/llama_3B_LR2e5_DR0.zip
replace content/llama_3B_LR2e5_DR0/tokenizer_config.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace content/llama_3B_LR2e5_DR0/special_tokens_map.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [ ]:
import pprint
import re
from pathlib import Path

import pandas as pd
import torch
from datasets import Dataset
from google.colab import files
from sklearn.model_selection import train_test_split
from transformers import TrainingArguments
from trl import DPOConfig, DPOTrainer

from unsloth import FastLanguageModel, PatchDPOTrainer, is_bfloat16_supported

In [ ]:
max_seq_length = 4096 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/content/llama_3B_LR2e5_DR0", # Choose ANY! eg mistralai/Mistral-7B-Instruct-v0.2
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

==((====))==  Unsloth 2024.12.4: Fast Llama patching. Transformers:4.46.3.
   \\   /|    GPU: NVIDIA A100-SXM4-40GB. Max memory: 39.564 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 8.0. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [31]:
instruction = "You are a tool used to extract the requirements and responsibilities from a job description. I will be inputting a series of job descriptions, so please use the following prompt:\n\nFirst, extract the technical skill requirements and responsibilities from the job description as JSON by following these steps:\nStep 1 - Technical skills should be defined as: specific abilities that relate to the use of tools, technologies, programming languages, software applications, or technical processes.\nStep 2 - Extract the technical skills that are explicitly stated in the job description. You must not include soft skills or job titles. You must include any technical skills that are listed as an experience requirement.\nStep 3 - Store each of the technical skills in a key called 'skill_name' under an object called “technical_skills”.\nStep 4 - If multiple technical skills in a phrase are grouped using parentheses or listed using grouping connectors similar to \"including\", \"like\", \"/\", \"or\", or \"such as\" then list the skills in one array so that they result in a single skill_name. If technical skills are listed without grouping connectors and in a phrase ending in \"and\" then treat each listed skill as a separate skill_name. When skills are listed with no clear grouping implied, list each skill individually rather than in an array.  Below are examples of desired outputs:\nExample phrase 1 -  \"experience of 6+ years using cybersecurity architectures, IDS/IPS, SIEM tools, or firewalls\"\nExample output 1 - skill_name: [cybersecurity architectures, IDS/IPS, SIEM tools, firewalls]\nExample phrase 2 - \"knowledge of database systems including MySQL, PostgreSQL, and Redis\" \nExample output 2 - skill_name: [database systems, MySQL, PostgreSQL, Redis]\nExample phrase 3  - \"4+ years experience with Java/JEE, SOAP/REST/Micro Services, XML\"\nExample output 3 (3 separate skill_name keys) - skill_name: [Java, JEE], skill_name: [SOAP, REST, Micro Services], skill_name: [XML] \nExample phrase 4  - \"experience with SQL Server, Oracle, MongoDB\" \nExample output 4 (3 separate skill_name keys) - skill_name: [SQL Server], skill_name: [Oracle], skill_name: [MongoDB]\nExample phrase 5 -  \"4 or more years experience with messaging architectures (EAI), SAP XI, BizTalk, and Web Services\" \nExample output 5 (4 separate skill_name keys) - skill_name: [messaging architectures, EAI], skill_name: [SAP XI], skill_name: [BizTalk], skill_name: [Web Services]\nExample phrase 6 - \"AWS Knowledge: Event Bridge, Cloud Watch, Cloud Trail\"\nExample output 6 (4 separate skill_name keys) - skill_name: [AWS], skill_name: [Event Bridge], skill_name: [Cloud Watch], skill_name: [Cloud Trail],\nExample phrase 7 - \"Working knowledge of EDB, Oracle, and AWS operations (S3, EC2).\"\nExample output 7 (3 separate skill_name keys) - skill_name: [EDB], skill_name: [Oracle], skill_name: [AWS Operations, S3, EC2]\nExample phrase 8 - \"expertise in the SAP Analytics Roadmap and new technologies such as SAP HANA, and SAP SAC.\"\nExample output 8 (2 separate skill_name keys) - skill_name: [SAP Analytics Roadmap], skill_name: [SAP HANA, SAP SAC]\nExample phrase 9 - \"Demonstrated understanding of WAN, LAN, and VPN networks, including the relationship between network infrastructure and IP-enabled devices.\"\nExample output 9 (4 separate skill_name keys) - skill_name: [WAN], skill_name: [LAN], skill_name: [VPN networks], skill_name: [network infrastructure, IP-enabled devices]\nStep 5 - Classify each of the technical skills that are described as mandatory, essential, needed, or similar as ‘required’. Classify the technical skills that are described as preferred, nice to have, or similar as ‘preferred’. If a requirement is not mentioned or not clear, then classify it as ‘preferred’. If a technical skill is only described under job responsibilities or job duties then classify it as 'preferred'. Separate the skills under two objects called ‘required’ and ‘preferred’.\nStep 6 - For each skill, extract the minimum years required explicitly stated for that specific skill (not inferred or through association) and list 0 if years are not explicitly stated for that specific skill. If a range of years is given, use the smallest year. Label these using a key called ‘minyears’.\nStep 7 - If any two technical skills you have extracted are extremely similar, merge them into a single skill_name array.\n\nNext, extract the required level of education from the job description as JSON by following these steps:\nStep 1 - Locate the requirements, qualifications, prerequisite, or a similar section of the job description and extract any educational requirements that are explicitly stated.\nStep 2 - Create a key called ‘education_level’ that lists the levels of education mentioned in the job qualifications. Categorize the levels of education under one of the following values that fits best: High School Diploma, Associate's, Current Bachelor's Student, Bachelor’s, Current Master's Student, Master’s, Doctorate, Postdoctorate, Vocational, None.\nStep 3 - Classify the levels of education that are explicitly described as mandatory, essential, required, or similar as ‘required’. Classify the levels of education that are explicitly described as preferred, nice to have, or similar as ‘preferred’.  If a qualification is not mentioned, then classify the level of education as ‘preferred’.  Separate these levels of education under two objects called ‘required’ and ‘preferred’.\nStep 4 - If more than one level of education meets the necessary requirement, classify the higher levels of education as 'preferred'. If multiple levels of education are preferred, store them all as one level of education using an array of comma separated values. As a result, there should only be one 'education_level' key in each of the 'required' and 'preferred' object. For example, phrases like \"A Bachelor's is required, a Master's/PhD is preferred\" would have Bachelor's under the 'required' object but [Master's, Doctorate] under the 'preferred' object.\nStep 5 - If experience is an acceptable alternative to an education requirement, add 'Or Experience' as a value into the 'education_level' array, but only if there is an existing education level. Below are examples of desired outputs:\nExample phrase 1 - \"a Master's degree in mathematics, STEM, or a related field, or 2 years of experience\"\nExample output 1 - education_level: [Master's, Or Experience] \nExample phrase 2- \" graduate in information systems or equivalent hands-on work experience\"\nExample output 2 - education_level: [Bachelor's, Or Experience] \nExample phrase 3 - \"**Option A**: Bachelor's or higher degree with at least 30 semester hours in mathematics and physical sciences.- **Option B**: Combination of education and experience equivalent to a 4-year course of study.\"\nExample output 3- education_level: [Current Bachelor's, Or Experience] \nStep 6 - For each level of education in the list, identify and include any fields of study mentioned in a key called ‘field_of_study’ as separate values in an array. If terms like  \"or similar\", \"or equivalent\", or \"or related\" are used, include ‘Related’ as a field of study. If no fields of study are listed then do not list anything. Below are examples of desired outputs:\nExample phrase 1 - \"A Bachelor's degree in Computer Science, STEM, or a related field\"\nExample output 1 - field_of_study: [Computer Science, STEM, Related]\nExample phrase 2 - \"A graduate in Computer Science, Business, STEM, or equivalent\"\nExample output 2 - field_of_study: [Computer Science, STEM, Related]\n\nNext extract any required credentials, certifications, licenses, or similar credentials from the job description as JSON by following these steps:\nStep 1 - Search within the job description for certifications, licenses, security clearances, or similar credential qualifications that are related to the job. \nStep 2 - Extract the names of any certifications, licenses, security clearances, or similar credentials that are explicitly stated under keys called ‘credential’. Do not include credential qualifications that are not related to the job like driver's licenses, citizenship documentation, or other unrelated credentials.\nStep 3 - If multiple options are acceptable, place them in an array under the credential. If both a credential and it's abbreviation/acronym are given, provide both in the array. Below are examples of desired outputs:\nExample phrase 1 - \"Certified as IAT or IAM Level III\"\nExample output 1- field_of_study:  [IAM Level III Certification, IAT Level III Certification]\nExample phrase 2 - \"CCNP (Cisco Certified Network Professional)\"\nExample output 2 - field_of_study:  [CCNP, Cisco Certified Network Professional]\nStep 4 - Classify each of these credentials that are explicitly described as mandatory, essential, required, or similar as ‘required’. Classify the credentials that are explicitly described as preferred, nice to have, or similar as ‘preferred’. If a qualification is not mentioned, then classify the credential as ‘preferred’. Separate these credentials under two objects called ‘required’ and ‘preferred’.\nStep 5 - If any certifications, licenses, security clearances, or similar credentials are listed as a technical skill, you must remove them from the technical skill section.\nStep 6 - The output should be structured under the key ‘Credentials’, under objects for ‘required’ and ‘preferred’.\n\nNext, extract the experience requirements from the job description as JSON by following these steps:\nStep 1 - Experience is defined as: The amount and type of prior work experience a candidate must have in order to qualify for the job or to complete the duties of the job. This includes the level of seniority, industry background, job responsibilities, and job duties but does not include technical skills.\nStep 2 - Identify and extract any experience requirements that are explicitly stated in the job description.\nStep 3 - Create a key for each experience requirement under a key called 'experience_desc'. \nStep 4 - Review the experience requirements and if any of them involve technical skills then you must remove them.\nStep 5 - If multiple experiences in a phrase are grouped using parentheses or listed using grouping connectors similar to \"including\", \"like\", \"/\", \"or\", or \"such as\" then you must list the experiences in one array so that they result in a single experience_desc. If experiences are listed without grouping connectors and listed in a phrase ending in \"and\" then you must treat each listed experience as a separate experience_desc. When no clear grouping is implied then you must list each experience individually rather than in an array. Below are examples of desired outputs:\nExample phrase 1 - \"7+ years overall experience in product management, marketing analysis, or professional services\"\nExample output 1 - experience_desc: [product management, marketing analysis, professional services]\nExample phrase 2 - \"must have leadership experience (manager, director, or VP level)\"\nExample output 2 - experience_desc: [manager, director, VP]\nExample phrase 3 - \"3+ years in designing, implementing, and optimizing custom solutions for enterprise software applications\"\nExample output 3 (3 separate experience_desc keys) - experience_desc: [designing custom solutions for enterprise software applications], experience_desc: [implementing custom solutions for enterprise software applications], experience_desc: [optimizing custom solutions for enterprise software applications]\nExample phrase 4 - \"1-2 years of experience leading software teams and working in an agile, kanban, or scrum environment\"\nExample output 4 (2 separate experience_desc keys) - experience_desc: [leading software teams], experience_desc: [agile environment, kanban environment, scrum environment]\nExample phrase 5 - \"professional experience with version/source control, design/application, development principles/methodologies, integration tools\"\nExample output 5 (4 separate experience_desc keys) - experience_desc: [version control, source control], experience_desc: [design, application], experience_desc: [development principles, development methodologies], experience_desc: [integration tools]  \nStep 6 - If terms like \"or similar\", \"or equivalent\", or \"or related\" are used to describe an experience requirement you must include ‘Related’ in the array.  Below are examples of desired outputs:\nExample phrase 1 - \"experience as a data scientist or a related role\"\nExample output 1 - experience_desc: [data scientist, Related]\nExample phrase 2 - \"5+ years of data engineering experience or equivalent\"\nExample output 2 - experience_desc: [data engineering, Related]\nStep 7 - If experience is listed as a requirement without specifying a type of experience, then list the experience as 'Work Experience'. Below are examples of desired outputs:\nExample phrase 1 - \"4+ years of experience are required\"\nExample output 1 - experience_desc: [Work Experience]\nExample phrase 2 - \"Experience Required: 5 years\"\nExample output 2 - experience_desc: [Work Experience]\nStep 8 - Identify the education requirements in the job description. For any phrasing that includes work experience as an alternative to an education requirement, such as \"equivalent experience\", \"equivalent hands-on work experience\", \"relevant experience\", or \"professional experience\" you must include an additional experience_desc key with the values [Work Experience, Or Education]. You must apply this rule to any phrasing where work experience is presented as an acceptable substitute for an education requirement. Below are examples of desired outputs:\nExample phrase 1 - \"An degree in engineering or 3+ years of relevant experience\"\nExample output 1 - experience_desc: [Work Experience, Or Education] \nExample phrase 2 - \"A Bachelor's degree in computer science (or equivalent experience) is needed\" \nExample output 2 - experience_desc: [Work Experience, Or Education] \nExample phrase 3 - \"Master of Science degree in computer science, MIS, or equivalent hands-on work experience\"\nExample output 3 - experience_desc: [Work Experience, Or Education] \nExample phrase 4 - \"A degree from an accredited university preferred, but related experience is also allowed\"\nExample output 4 - experience_desc: [Work Experience, Or Education] \nStep 9 - For each experience_desc in the list, identify and include any industries mentioned in a key called ‘industry’ as separate values in an array. If terms like  \"or similar\", \"or equivalent\", or \"or related\" are used, include ‘Related’ as an industry.  If no industries are listed then do not list anything. Below are examples of desired outputs:\nExample phrase 1 - \"Minimum of 5+ years of experience as a Project Manager / Scrum Master in the AI/ML industry\"\nExample output 1 - industry: [AI Industry, ML Industry]\nExample phrase 2 - \"7 years of data analyst experience in SaaS\"\nExample output 2 - industry: [SaaS]\nExample phrase 3 - \"Proven leadership experience in fintech or a related industry\"\nExample output 3 - industry: [fintech, Related]\nStep 10 - Classify each of the experiences that are explicitly described as mandatory, essential, required, or similar as ‘required’. Classify the experiences that are explicitly described as preferred, nice to have, or similar as ‘preferred’. If a requirement is not mentioned or not clear, then classify the experience as ‘preferred’. If an experience is only described under job responsibilities or job duties then classify it as 'preferred'. Separate these experiences under two objects called ‘required’ and ‘preferred’.\nStep 11 - Review each experience, and if any experiences contain technical skills within them remove them from the experience requirements and then add them to the technical skills requirements.\nStep 12 - If any industries under 'required' are described as preferred then remove the industry.\nExample phrase 1 - \"2+ years of experience in Information Technology, preferably in the semiconductor industry\"\nExample output 1 - industry: []\nExample phrase 2 - \"4 years of experience as a data engineer are required, within the finance industry preferred\"\nExample output 2 - industry: []\nStep 13 - For each experience, list the minimum years of experience that is explicitly specified for that experience and list 0 if years are not explicitly specified. If a range of years is given, give the smallest year. Label these using a key called ‘minyears’.\n\nLastly, clean and organize the extracted requirements and responsibilities by following these steps:\nStep 1 - If the job description contains the requirements for multiple positions or multiple levels of a position, you must only keep the 'required' qualifications that are relevant to the lowest position or most entry level position and then treat the rest of the qualifications as 'preferred'.\nStep 2 - Combine the JSON outputs from skills, education, credentials, and experience into one JSON. \nStep 3 - Organize the data under two main objects: 'required' and 'preferred', moving the respective skills, education, credentials, and experience under them accordingly. \nStep 4 - Output the combined JSON."

In [ ]:

def format_for_dpo(example, tokenizer, assistant_prefix="<|assistant|>\n"):
    def _strip_prefix(s, pattern):
        # Use re.escape to escape any special characters in the pattern
        return re.sub(f"^{re.escape(pattern)}", "", s)

    chosen_messages = [
        {"role": "user", "content": example["chosen"]}
    ]
    rejected_messages = [
        {"role": "user", "content": example["rejected"]}
    ]
    prompt_messages = [
        {"role": "system", "content": instruction },
        {"role": "user", "content": example["description"]},
    ]

    example["text_chosen"] = tokenizer.apply_chat_template(chosen_messages, tokenize=False)
    example["text_rejected"] = tokenizer.apply_chat_template(rejected_messages, tokenize=False)
    example["text_prompt"] = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    example["text_chosen"] = _strip_prefix(example["text_chosen"], assistant_prefix)
    example["text_rejected"] = _strip_prefix(example["text_rejected"], assistant_prefix)

    return example

In [ ]:
data_path = Path("/content/threshold_dpo_data_new_scoring.xlsx")
threshold_df = pd.read_excel(data_path.as_posix())
print(threshold_df.shape)
threshold_df.head()

train_df, val_df = train_test_split(threshold_df, test_size=0.3, random_state=42)

(1804, 52)


In [34]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

In [35]:
column_names = train_dataset.features

train_dataset = train_dataset.map(
    format_for_dpo,
    fn_kwargs={"tokenizer": tokenizer},
    num_proc=12,
    remove_columns=column_names,
    desc="Formatting comparisons with prompt template",
)

Formatting comparisons with prompt template (num_proc=12):   0%|          | 0/1262 [00:00<?, ? examples/s]

In [36]:
train_dataset

Dataset({
    features: ['text_chosen', 'text_rejected', 'text_prompt'],
    num_rows: 1262
})

In [37]:
column_names = val_dataset.features

val_dataset = val_dataset.map(
    format_for_dpo,
    fn_kwargs={"tokenizer": tokenizer},
    num_proc=12,
    remove_columns=column_names,
    desc="Formatting comparisons with prompt template",
)

Formatting comparisons with prompt template (num_proc=12):   0%|          | 0/542 [00:00<?, ? examples/s]

In [39]:
val_dataset

Dataset({
    features: ['text_chosen', 'text_rejected', 'text_prompt'],
    num_rows: 542
})

In [40]:
train_dataset = train_dataset.rename_columns(
        {"text_prompt": "prompt", "text_chosen": "chosen", "text_rejected": "rejected"}
)

val_dataset = val_dataset.rename_columns(
        {"text_prompt": "prompt", "text_chosen": "chosen", "text_rejected": "rejected"}
)

We shall print a random item from the dataset

In [ ]:
row = train_dataset[8]
pprint.pprint(row["prompt"])
pprint.pprint(row["chosen"])
pprint.pprint(row["rejected"])

('<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n'
 '\n'
 'Cutting Knowledge Date: December 2023\n'
 'Today Date: 07 Dec 2024\n'
 '\n'
 'You are a tool used to extract the requirements and responsibilities from a '
 'job description. I will be inputting a series of job descriptions, so please '
 'use the following prompt:\n'
 '\n'
 'First, extract the technical skill requirements and responsibilities from '
 'the job description as JSON by following these steps:\n'
 'Step 1 - Technical skills should be defined as: specific abilities that '
 'relate to the use of tools, technologies, programming languages, software '
 'applications, or technical processes.\n'
 'Step 2 - Extract the technical skills that are explicitly stated in the job '
 'description. You must not include soft skills or job titles. You must '
 'include any technical skills that are listed as an experience requirement.\n'
 "Step 3 - Store each of the technical skills in a key called 'skill_name' "
 'under 

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [20]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 42,
    use_rslora = True,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Already have LoRA adapters! We shall skip this step.


In [ ]:
PatchDPOTrainer()
dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,
    args = DPOConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 2,
        learning_rate = 5e-7,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,

        evaluation_strategy="steps",  # Evaluate during training at regular intervals
        eval_steps=10,               # Perform evaluation every 10 steps
        save_strategy="steps",       # Save checkpoints at every evaluation
        save_steps=10,               # Same as eval_steps for simplicity
        save_total_limit=2,          # Retain only 2 checkpoints
        load_best_model_at_end=True, # Automatically load the best model after training
        metric_for_best_model="eval_loss",  # Use validation loss to determine the best model
        greater_is_better=False,

        optim = "adamw_8bit",
        weight_decay = 0.0,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
    beta = 0.1,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    tokenizer = tokenizer,
    max_length = 1024,
    max_prompt_length = 512,
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Extracting prompt from train dataset:   0%|          | 0/1262 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/1262 [00:00<?, ? examples/s]

Extracting prompt from eval dataset:   0%|          | 0/542 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/542 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1262 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/542 [00:00<?, ? examples/s]

In [47]:
dpo_trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 1,262 | Num Epochs = 2
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 4
\        /    Total batch size = 8 | Total steps = 314
 "-____-"     Number of trainable parameters = 24,313,856


Step,Training Loss,Validation Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / rejected,logps / chosen,logits / rejected,logits / chosen
10,0.743300,1.111243,7.413105,7.537846,0.473652,-0.124742,-359.991699,-372.360413,5.161123,5.158449
20,0.674600,1.101816,7.305326,7.423288,0.468137,-0.117962,-361.137299,-373.438202,5.162179,5.159704
30,0.752400,1.089748,7.187899,7.291922,0.475490,-0.104024,-362.450989,-374.612488,5.165170,5.162203
40,0.987600,1.077302,7.161669,7.250747,0.477328,-0.089078,-362.862701,-374.874756,5.165324,5.162570
50,1.317600,1.073535,7.194562,7.277875,0.477328,-0.083313,-362.591431,-374.545807,5.164906,5.162005
60,0.776200,1.064566,7.260928,7.332584,0.481005,-0.071658,-362.044342,-373.882172,5.166181,5.163336
70,1.210000,1.059772,7.270380,7.335848,0.475490,-0.065468,-362.011719,-373.787659,5.163095,5.160412
80,1.821900,1.050038,7.313663,7.363576,0.479167,-0.049914,-361.734436,-373.354828,5.162731,5.159635
90,1.353300,1.041509,7.258608,7.299901,0.484681,-0.041292,-362.371155,-373.905396,5.160435,5.157497
100,0.372000,1.033373,7.232646,7.263826,0.486520,-0.031181,-362.731934,-374.164978,5.158610,5.155633


TrainOutput(global_step=314, training_loss=1.0328878593292965, metrics={'train_runtime': 4121.6072, 'train_samples_per_second': 0.612, 'train_steps_per_second': 0.076, 'total_flos': 0.0, 'train_loss': 1.0328878593292965, 'epoch': 1.9920760697305864})

In [48]:
model.save_pretrained("dpo_model_5e7_b01")
tokenizer.save_pretrained("dpo_tokenizer_5e7_b01")

('dpo_tokenizer_5e7_b01/tokenizer_config.json',
 'dpo_tokenizer_5e7_b01/special_tokens_map.json',
 'dpo_tokenizer_5e7_b01/tokenizer.json')

In [49]:
!zip -r dpo_model_5e7_b01.zip dpo_model_5e7_b01

  adding: dpo_model_5e7_b01/ (stored 0%)
  adding: dpo_model_5e7_b01/adapter_config.json (deflated 53%)
  adding: dpo_model_5e7_b01/README.md (deflated 66%)
  adding: dpo_model_5e7_b01/adapter_model.safetensors (deflated 7%)


In [ ]:
files.download('dpo_model_5e7_b01.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>